In [ ]:
import numpy as np
from scipy.optimize import fsolve
from scipy.interpolate import UnivariateSpline
from matplotlib import pyplot as plt 
from scipy.spatial.distance import euclidean

In [ ]:
circle_center = np.array([1.5, 0])  # Center of the circle (cx, cy)
radius = 0.75  # Radius of the circle

In [ ]:
x = np.arange(0, 3, 0.001)

In [ ]:
# y = 1-np.exp(-x)
y = np.sin(6*np.pi*x)

In [ ]:
def plot_fig(pts=None):
    fig, ax = plt.subplots()
    ax.plot(x,y)
    circle = plt.Circle(circle_center, radius, color="r", fill=False)
    ax.add_patch(circle)
    ax.set_aspect(1)

    if pts is not None:
        for pt in pts:
            plt.scatter(*pt)
    ax.grid()

In [ ]:
plot_fig()

In [ ]:
o = (0,0)
p = np.stack([x,y],axis=-1)
t = np.cumsum(np.sqrt(np.sum(np.diff(p, axis=0) ** 2, axis=1)))
t = np.insert(t,0,0) / t[-1]

In [ ]:
t

In [ ]:
spline_x = UnivariateSpline(t, x, s=len(t)*0.001)
spline_y = UnivariateSpline(t, y, s=len(t) * 0.001)

In [ ]:
def distance_to_circle_center(t):
    x, y = spline_x(t).item(), spline_y(t).item()
    point_on_curve = np.array([x, y])  # Point on the curve
    dist = euclidean(point_on_curve, circle_center)

    return dist - radius

In [ ]:
dd = [distance_to_circle_center(e) for e in t]
plt.plot(t, dd)

In [ ]:
initial_t_guesses = np.linspace(0, 1, 100)

In [ ]:
t_intersection = np.array(
    [fsolve(distance_to_circle_center, t_guess)[0] for t_guess in initial_t_guesses]
)

In [ ]:
t_intersection = [t for t in t_intersection if np.isclose(distance_to_circle_center(t), 0)]

In [ ]:
t_intersection_unique = np.unique(np.round(t_intersection, 5))
t_intersection_unique = t_intersection_unique[
    (t_intersection_unique >= 0) & (t_intersection_unique <= 1)
]

In [ ]:
intersection_points = np.array(
    [[spline_x(t), spline_y(t)] for t in t_intersection_unique]
)

In [ ]:
print("Intersection points:", intersection_points)

In [ ]:
plot_fig(intersection_points)